In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from IPython.display import display

# --- Load vocab ---
vocab10K = pd.read_csv("vocab10K.csv")

# --- Load X safely ---
npz = np.load("X_word_count.npz", allow_pickle=True)
X = npz["X"]
if isinstance(X, np.ndarray) and X.dtype == object:
    X = X.item()

# --- Load keys + posts and align rows ---
posts = pd.read_csv("gendered_posts.csv")
keys  = pd.read_csv("keys_to_X.csv")

keys2 = keys.reset_index().rename(columns={"index": "row_id"})
merged = keys2.merge(posts, on=["title_id", "post_id"], how="inner")

row_id = merged["row_id"].to_numpy()
y = merged["female"].astype(int).to_numpy()

X_sub = X[row_id, :]

print("X_sub:", X_sub.shape, " y:", y.shape)

# --- OLS-style fit (Ridge ~ OLS) ---
ols = Ridge(alpha=1e-6, fit_intercept=True)
ols.fit(X_sub, y)
coef = ols.coef_

# --- Attach coefficients and build Table 1 ---
vocab10K["ME_ols"] = coef

top_female_ols = (
    vocab10K.sort_values("ME_ols", ascending=False)[["word", "ME_ols"]]
    .head(10).reset_index(drop=True)
)
top_male_ols = (
    vocab10K.sort_values("ME_ols", ascending=True)[["word", "ME_ols"]]
    .head(10).reset_index(drop=True)
)

tab1_ols = pd.concat([top_female_ols, top_male_ols], axis=1)
tab1_ols.columns = pd.MultiIndex.from_tuples([
    ("Most female", "Word"), ("Most female", "ME (OLS)"),
    ("Most male",   "Word"), ("Most male",   "ME (OLS)")
])

sty_ols = (
    tab1_ols.style
      .format({("Most female","ME (OLS)"): "{:.3f}",
               ("Most male","ME (OLS)"): "{:.3f}"})
      .set_caption("Table 1 (OLS)—Top 10 Words Most Predictive of Female/Male Posts")
      .set_table_styles([
          {"selector": "caption", "props": [
              ("caption-side","top"), ("font-weight","bold"),
              ("text-align","center"), ("margin-bottom","10px")]},
          {"selector": "th", "props": [
              ("text-align","center"),
              ("border-top","2px solid black"),
              ("border-bottom","1px solid black")]},
          {"selector": "td", "props": [("text-align","center")]},
          {"selector": "table", "props": [
              ("border-bottom","2px solid black"),
              ("margin-left","auto"), ("margin-right","auto")]}
      ])
      .hide(axis="index")
)

display(sty_ols)


FileNotFoundError: [Errno 2] No such file or directory: 'vocab10K.csv'